# Week 8 — Model Evaluation and Selection

**File:** `04_model_evaluation_week8.ipynb`

This notebook loads the processed IPL dataset and every model saved during Week 7. It evaluates the models on the same held-out test set, creates comparison plots, creates confusion matrices by converting continuous performance scores into categories, checks whether ROC curves are applicable, and saves the final model-selection results.

---
## CODE REVIEW SUMMARY
**Reviewer:** Michael (Group 8, 7PAM2033)  
**Reviewed Team:** Group 1  
**Reviewed:** `04_model_evaluation_week8.ipynb`

A well-organised and thoughtfully written evaluation notebook. The pipeline loads the processed dataset and every model saved from Week 7, evaluates them consistently on the same held-out test set, and correctly handles the fact that these are regression models by converting scores into categories only for supplementary classification-style reporting. The path-resolution logic and the leak-free category thresholds both show genuinely careful engineering. See inline review comments (marked with `> **CODE REVIEW**`) throughout this notebook for specific feedback.

**Priority 1 (Must Do):** Add a docstring to `score_to_category()`  
**Priority 2 (Should Do):** Clearer error message for feature/column mismatch; confirm random seed consistency between Week 7 and Week 8  
**Priority 3 (Nice to Have):** Comment on `np.select` condition order; reduce repeated dictionary lookup; trust-boundary comment on `joblib.load`
---

> **CODE REVIEW (ADDITIONAL) — Reviewer: Michael (Group 8)**
> 
> **[Coding Standards | Positive]** Checking this notebook's naming against the Test Plan's (Section 2) stated conventions: the file itself follows `lowercase_with_underscores` naming (`04_model_evaluation_week8.ipynb`), and every variable throughout — `evaluation_rows`, `predictions_by_model`, `best_model_name`, `category_labels` — correctly uses `lowercase_with_underscores` as specified. No PascalCase or camelCase slip-ups found anywhere in this notebook, which is a good sign of consistency across the team's codebase.

## 1. Imports

In [ ]:
# Install missing packages only when required:
# !pip install pandas numpy matplotlib scikit-learn xgboost lightgbm joblib

import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
print("Imports completed.")

> **CODE REVIEW — Reviewer: Michael (Group 8)**
> 
> **[Logic & Readability | Positive]** Really good piece of engineering — this correctly handles the notebook being opened from either `/notebooks` or the project root, which is a portability issue that has tripped up other notebooks reviewed this term. No changes needed here, just wanted to flag this as a strong pattern worth reusing elsewhere.

In [ ]:
from pathlib import Path

# Find the repository root whether the notebook is opened from /notebooks or the project root.
CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
if not (PROJECT_ROOT / "data" / "processed" / "ipl_features.csv").exists():
    candidates = list(CURRENT.rglob("data/processed/ipl_features.csv"))
    if not candidates:
        raise FileNotFoundError("Could not find data/processed/ipl_features.csv")
    PROJECT_ROOT = candidates[0].parents[2]

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ipl_features.csv"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
PLOTS_DIR = RESULTS_DIR / "plots"

for folder in [MODELS_DIR, RESULTS_DIR, PLOTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")

## 2. Load processed dataset and training metadata

> **CODE REVIEW — Reviewer: Michael (Group 8)**
> 
> **[Functionality | Minor]** If `metadata["features"]` ever references a column that no longer exists in `ipl_features.csv` (e.g. after a Week 7 feature engineering change), the `model_data = df[feature_columns + [TARGET]]` line below will raise a `KeyError` with a fairly unhelpful message. Consider adding a check before that line:
> ```python
> missing = set(feature_columns) - set(df.columns)
> if missing:
>     raise ValueError(f"Missing expected feature columns: {missing}")
> ```
> This gives a much clearer error message and aligns with the Test Plan's (Section 2) requirement for error handling when data is unavailable or malformed.

> **CODE REVIEW (ADDITIONAL) — Reviewer: Michael (Group 8)**
> 
> **[Security & Performance | Minor]** `feature_columns` and `TARGET` below are read from `training_metadata.json` and used directly to index into the DataFrame with no validation that these are expected values. Low risk within your own private repo, but validating that `feature_columns` is a list of strings and `TARGET` is a single string (rather than trusting the file contents blindly) would harden this against a corrupted or tampered metadata file.

In [ ]:
metadata_path = RESULTS_DIR / "training_metadata.json"
if not metadata_path.exists():
    raise FileNotFoundError(
        "results/training_metadata.json was not found. Run 03_model_training_week7.ipynb first."
    )

with open(metadata_path, "r", encoding="utf-8") as file:
    metadata = json.load(file)

TARGET = metadata["target"]
feature_columns = metadata["features"]
RANDOM_STATE = metadata["random_state"]
TEST_SIZE = metadata["test_size"]

df = pd.read_csv(DATA_PATH)
model_data = df[feature_columns + [TARGET]].replace([np.inf, -np.inf], np.nan)
model_data = model_data.dropna(subset=[TARGET]).reset_index(drop=True)

X = model_data[feature_columns]
y = model_data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"Dataset shape: {df.shape}")
print(f"Evaluation rows: {len(X_test)}")
print(f"Features: {len(feature_columns)}")

## 3. Load all saved models

> **CODE REVIEW — Reviewer: Michael (Group 8)**
> 
> **[Security & Performance | Minor]** Loading five `.pkl` files via `joblib.load` below is fine since the codebase controls what gets saved, but pickle-based formats can execute arbitrary code if ever loaded from an untrusted source. Low risk here since it's your own private repo, but a short comment noting that models should only be loaded from the trusted `models/` directory would be good practice given the documentation standard in the Test Plan (Section 2).

In [ ]:
model_names = ["random_forest", "xgboost", "mlp", "hybrid", "lightgbm"]
loaded_models = {}

for name in model_names:
    model_path = MODELS_DIR / f"{name}.pkl"
    if not model_path.exists():
        raise FileNotFoundError(f"Missing {model_path}. Run the Week 7 notebook first.")
    loaded_models[name] = joblib.load(model_path)
    print(f"Loaded: {model_path.name}")

> **CODE REVIEW (ADDITIONAL) — Reviewer: Michael (Group 8)**
> 
> **[Security & Performance | Minor]** All 5 models are loaded into memory simultaneously via `joblib.load` below. For larger models (e.g. if the hybrid model grows to include a full neural network), this could become a memory concern. Consider whether models could be loaded and evaluated one at a time, releasing each from memory after use, if this pipeline is ever scaled up.

## 4. Evaluate each regression model

In [ ]:
evaluation_rows = []
predictions_by_model = {}

for name, model in loaded_models.items():
    predictions = model.predict(X_test)
    predictions_by_model[name] = predictions
    evaluation_rows.append({
        "model": name,
        "r2": r2_score(y_test, predictions),
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": mean_squared_error(y_test, predictions) ** 0.5,
    })

evaluation_df = pd.DataFrame(evaluation_rows).sort_values(
    ["rmse", "mae"], ascending=[True, True]
).reset_index(drop=True)
display(evaluation_df)

> **CODE REVIEW (ADDITIONAL) — Reviewer: Michael (Group 8)**
> 
> **[Security & Performance | Minor]** No inference timing is captured in this evaluation loop. It would be valuable to record how long `model.predict(X_test)` takes per model (e.g. using `time.perf_counter()`) alongside R²/MAE/RMSE, since prediction latency could matter if the final application needs to serve predictions live.

## 5. Generate confusion matrices using performance categories

> **CODE REVIEW — Reviewer: Michael (Group 8)**
> 
> **[Functionality | Positive]** Computing `q1, q2` from `y_train` only (below) avoids leaking test-set information into the category boundaries. This directly supports the Test Plan's (Section 5) "no data leakage through time-based splitting" KPI. Nice catch.
>
> **[Coding Standards | Minor]** The comment above `score_to_category()` explains the purpose well, but the Test Plan (Section 2) requires docstrings on important functions specifically. Consider adding:
> ```python
> def score_to_category(values):
>     """Convert continuous performance scores into Low/Average/High categories
>     using quantile thresholds computed from the training set only."""
> ```
>
> **[Logic & Readability | Minor]** A one-line comment above the `np.select(...)` call clarifying which condition maps to which label would help a future reader quickly verify the mapping, e.g.: `# <= q1 -> Low, <= q2 -> Average, otherwise -> High`

In [ ]:
# Confusion matrices are classification tools. Because these models predict a continuous
# score, scores are converted into three categories using thresholds calculated only
# from the training target. This avoids using test-set information to define categories.
q1, q2 = y_train.quantile([0.33, 0.66]).tolist()
category_labels = ["Low", "Average", "High"]

def score_to_category(values):
    values = np.asarray(values)
    return np.select(
        [values <= q1, values <= q2],
        ["Low", "Average"],
        default="High",
    )

y_test_category = score_to_category(y_test)
classification_rows = []

for name, predictions in predictions_by_model.items():
    predicted_category = score_to_category(predictions)
    matrix = confusion_matrix(y_test_category, predicted_category, labels=category_labels)

    display_obj = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=category_labels,
    )
    display_obj.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix — {name.replace('_', ' ').title()}")
    plt.tight_layout()
    path = PLOTS_DIR / f"confusion_matrix_{name}.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()

    report = classification_report(
        y_test_category,
        predicted_category,
        labels=category_labels,
        output_dict=True,
        zero_division=0,
    )
    classification_rows.append({
        "model": name,
        "category_accuracy": report["accuracy"],
        "macro_f1": report["macro avg"]["f1-score"],
    })

classification_df = pd.DataFrame(classification_rows).sort_values(
    "macro_f1", ascending=False
).reset_index(drop=True)
display(classification_df)

> **CODE REVIEW (ADDITIONAL) — Reviewer: Michael (Group 8)**
> 
> **[Security & Performance | Minor]** `plt.show()` is called inside this loop for each of the 5 models, but figures are never explicitly closed with `plt.close()` after saving. This can cause matplotlib to hold multiple open figures in memory simultaneously. Adding `plt.close()` after each `plt.savefig()` call would free memory as each plot completes, which matters more as the number of models compared grows.

## 6. Plot ROC curves if applicable

> **CODE REVIEW — Reviewer: Michael (Group 8)**
> 
> **[Functionality | Positive]** Good handling of an edge case — rather than silently skipping ROC curves or crashing when trying to call `predict_proba` on a regressor, this explicitly checks capability first and prints a clear explanation. This is exactly the kind of edge case awareness the Test Plan (Section 4) calls for in unit testing.

In [ ]:
# ROC curves normally require classification probabilities from predict_proba or
# decision_function. The saved models are regressors and return continuous performance
# scores, so a conventional ROC curve is not applicable and is intentionally not plotted.
roc_capable = {
    name: hasattr(model, "predict_proba") or hasattr(model, "decision_function")
    for name, model in loaded_models.items()
}
print("ROC capability:", roc_capable)
if not any(roc_capable.values()):
    print("ROC curves skipped: all saved models are regression models.")

## 7. Compare all models

In [ ]:
comparison_df = evaluation_df.merge(classification_df, on="model", how="left")
display(comparison_df)

comparison_df.to_csv(RESULTS_DIR / "week8_model_comparison.csv", index=False)
with open(RESULTS_DIR / "week8_model_comparison.json", "w", encoding="utf-8") as file:
    json.dump(comparison_df.to_dict(orient="records"), file, indent=2)

print("Saved final comparison metrics.")

In [ ]:
# Regression metric comparison
plot_df = comparison_df.sort_values("rmse", ascending=False)
ax = plot_df.plot(x="model", y=["rmse", "mae"], kind="bar", figsize=(10, 5))
ax.set_title("Final Regression Model Comparison")
ax.set_xlabel("Model")
ax.set_ylabel("Error")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
path = PLOTS_DIR / "week8_regression_model_comparison.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {path}")

In [ ]:
# R² comparison
plot_df = comparison_df.sort_values("r2")
ax = plot_df.plot(x="model", y="r2", kind="bar", legend=False, figsize=(9, 5))
ax.set_title("Final R² Comparison")
ax.set_xlabel("Model")
ax.set_ylabel("R² (higher is better)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
path = PLOTS_DIR / "week8_r2_comparison.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {path}")

## 8. Select and save the best model

> **CODE REVIEW — Reviewer: Michael (Group 8)**
> 
> **[Security & Performance | Minor]** `RANDOM_STATE` is correctly pulled from the saved metadata for the train/test split, which is good practice. Worth double-checking this same `RANDOM_STATE` was used consistently in the Week 7 training notebook for any random elements inside model training itself (e.g. Random Forest, XGBoost), so that these final performance figures are fully reproducible end to end.

In [ ]:
# Primary selection criterion: lowest RMSE. MAE is used as the tie-breaker.
best_row = comparison_df.sort_values(["rmse", "mae"], ascending=[True, True]).iloc[0]
best_model_name = best_row["model"]
best_model = loaded_models[best_model_name]

joblib.dump(best_model, MODELS_DIR / "best_model_week8.pkl")

final_metrics = {
    "best_model": best_model_name,
    "selection_rule": "Lowest test RMSE, with MAE as tie-breaker",
    "r2": float(best_row["r2"]),
    "mae": float(best_row["mae"]),
    "rmse": float(best_row["rmse"]),
    "category_accuracy": float(best_row["category_accuracy"]),
    "macro_f1": float(best_row["macro_f1"]),
    "category_thresholds": {"low_to_average": float(q1), "average_to_high": float(q2)},
}
with open(RESULTS_DIR / "final_model_metrics.json", "w", encoding="utf-8") as file:
    json.dump(final_metrics, file, indent=2)

print(f"Best model: {best_model_name}")
print(f"RMSE: {best_row['rmse']:.4f}")
print(f"MAE: {best_row['mae']:.4f}")
print(f"R²: {best_row['r2']:.4f}")
print("Saved models/best_model_week8.pkl")
print("Saved results/final_model_metrics.json")

## 9. Actual versus predicted plot for the selected model

> **CODE REVIEW — Reviewer: Michael (Group 8)**
> 
> **[Logic & Readability | Minor]** `predictions_by_model[best_model_name]` below is a repeated dictionary lookup similar to the pattern in the prior cell. Not a performance concern given the small dictionary size, but assigning it once right after best model selection would slightly reduce repetition.

In [ ]:
best_predictions = predictions_by_model[best_model_name]
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.65)
minimum = min(float(y_test.min()), float(np.min(best_predictions)))
maximum = max(float(y_test.max()), float(np.max(best_predictions)))
plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")
plt.title(f"Actual vs Predicted — {best_model_name.replace('_', ' ').title()}")
plt.xlabel("Actual Performance Score")
plt.ylabel("Predicted Performance Score")
plt.tight_layout()
path = PLOTS_DIR / "week8_best_model_actual_vs_predicted.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {path}")

> **CODE REVIEW (ADDITIONAL) — Reviewer: Michael (Group 8)**
> 
> **[Coding Standards | Minor]** The Test Plan (Section 2) describes a modularised codebase with separate files for Data Cleaning, Feature Engineering, Model Building, Model Evaluation, Prediction Making, and Application Development. This notebook correctly represents the Model Evaluation stage on its own, which is good. One suggestion: the notebook currently mixes several sub-tasks (evaluation, confusion matrices, ROC checking, comparison, best model selection, plotting) within a single notebook using markdown headers rather than separate functions. Wrapping each of these steps into a named function (e.g. `evaluate_models()`, `generate_confusion_matrices()`, `select_best_model()`) defined in a small companion `.py` module would make each step independently unit-testable via Pytest, which the Test Plan (Section 4) specifically calls for.

## 10. Week 8 outputs

- `models/best_model_week8.pkl`
- `results/week8_model_comparison.csv`
- `results/week8_model_comparison.json`
- `results/final_model_metrics.json`
- `results/plots/confusion_matrix_*.png`
- `results/plots/week8_regression_model_comparison.png`
- `results/plots/week8_r2_comparison.png`
- `results/plots/week8_best_model_actual_vs_predicted.png`